In [9]:
# ! pip install scipy

In [10]:
# ! pip install implicit

In [11]:
# ! pip install surprise

In [1]:
import gzip
import random
import scipy
import numpy
from collections import defaultdict
from implicit import bpr
from surprise import SVD, Reader, Dataset
from surprise.model_selection import train_test_split
from tqdm import tqdm as tqdm

/opt/miniconda3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
! pwd

/Users/hrithik/Desktop/UC San Diego/Courses/UCSD CSE 258/Assignment 1/assignment1_2026


In [3]:
dataDir = "/Users/hrithik/Desktop/UC San Diego/Courses/UCSD CSE 258/Assignment 1/assignment1_2026"

In [18]:
def latentFactorModel(trainFile):
    a = open(trainFile, 'r')
    b = open("temp.csv", 'w')
    a.readline()
    for l in a.readlines():
        b.write(l)
    a.close()
    b.close() # Need to delete the header
    
    reader = Reader(line_format='user item timestamp rating', sep=',')
    data = Dataset.load_from_file("temp.csv", reader=reader)
    
    model = SVD(n_factors=32, n_epochs=50)
    trainset, testset = train_test_split(data, test_size=.25)
    model.fit(trainset)
    
    predictions = model.test(testset)
    
    sse = 0
    for p in predictions:
        sse += (p.r_ui - p.est)**2
    
    print("Error on own test set = ", str(sse / len(predictions)))
    
    return trainset, model

In [19]:
def testPredictionsLFM(trainset, model, testIn, testOut):
    a = open(testIn, 'r')
    b = open(testOut, 'w')
    header = a.readline()
    b.write(header)
    for l in a.readlines():
        fields = l.split(',')
        u,i = fields[0].strip(),fields[1].strip()
        try:
            #print((u,i))
            est = model.predict(u,i).est #Ratings
            b.write(l.strip() + ',' + str(est) + '\n')
        except Exception as e:
            print(e)
    a.close()
    b.close()

In [20]:
def BPRModel(trainFile):
    a = open(trainFile, 'r')
    a.readline() # header
    
    data = []
    userIDs,itemIDs = {},{}
    
    for l in a.readlines():
        fields = l.split(',')
        u,i = fields[0].strip(),fields[1].strip()
        data.append((u,i))
        if not u in userIDs: userIDs[u] = len(userIDs)
        if not i in itemIDs: itemIDs[i] = len(itemIDs)
        
    nUsers,nItems = len(userIDs),len(itemIDs)

    Xui = scipy.sparse.lil_matrix((nUsers, nItems))
    for u,i in data:
        Xui[userIDs[u],itemIDs[i]] = 1

    Xui_csr = scipy.sparse.csr_matrix(Xui)
    
    model = bpr.BayesianPersonalizedRanking(factors = 5)
    
    model.fit(Xui_csr)
    return model, userIDs, itemIDs

In [21]:
def testPredictionsBPR(model, userIDs, itemIDs, testIn, testOut):
    a = open(testIn, 'r')
    b = open(testOut, 'w')
    header = a.readline()
    b.write(header)
    randomGuesses = 0
    for l in a.readlines():
        fields = l.split(',')
        # User is first field, items are second last and last fields
        u,i1,i2 = fields[0].strip(),fields[-2].strip(),fields[-1].strip()
        try:
            u_ = userIDs[u]
            i1_ = itemIDs[i1]
            i2_ = itemIDs[i2]
        except Exception as e:
            #print("Missing user or item, guessing randomly")
            b.write(l.strip() + ',' + random.choice([i1,i2]) + '\n')
            randomGuesses += 1
        try:
            uf = model.user_factors[u_]
            i1f = model.item_factors[i1_]
            i2f = model.item_factors[i2_]
            est = i1
            if uf.dot(i2f) > uf.dot(i1f):
                est = i2
            b.write(l.strip() + ',' + est + '\n')
        except Exception as e:
            print(e)
    a.close()
    b.close()
    print("Guessed randomly " + str(randomGuesses) + " times")

In [22]:
# Task 1: latent factor model

In [23]:
trainset,m1 = latentFactorModel(dataDir + '/' + "tasks_1_2_train.csv")

Error on own test set =  0.862885193686915


In [17]:
testPredictionsLFM(trainset, m1, "task_1_test.csv", "task_1_predictions.csv")

In [ ]:
# Task 2: BPR

In [30]:
m2,userIDs,itemIDs = BPRModel(dataDir + '/' + "tasks_1_2_train.csv")

100%|██████████| 100/100 [00:13<00:00,  7.37it/s, train_auc=89.82%, skipped=1.97%]


In [31]:
testPredictionsBPR(m2, userIDs, itemIDs, "task_2_test.csv", "task_2_predictions.csv")

Guessed randomly 0 times


In [32]:
# Task 3: latent factor model, just ignore the timestamp

In [33]:
trainset,m3 = latentFactorModel(dataDir + '/' + "tasks_3_4_train.csv")

Error on own test set =  0.94363490951933


In [34]:
testPredictionsLFM(trainset, m3, "task_3_test.csv", "task_3_predictions.csv") # Don't use the timestamp

In [35]:
# Task 4: BPR, just ignore the timestamp

In [36]:
m4,userIDs,itemIDs = BPRModel(dataDir + '/' + "tasks_3_4_train.csv")

100%|██████████| 100/100 [00:09<00:00, 10.11it/s, train_auc=84.25%, skipped=5.14%]


In [37]:
testPredictionsBPR(m4, userIDs, itemIDs, "task_4_test.csv", "task_4_predictions.csv")

Guessed randomly 0 times


In [38]:
# Task 5: BPR, ignore timestamp/ordering

In [39]:
m5,userIDs,itemIDs = BPRModel(dataDir + '/' + "task_5_train.csv")

100%|██████████| 100/100 [00:27<00:00,  3.67it/s, train_auc=89.80%, skipped=1.84%]


In [40]:
testPredictionsBPR(m5, userIDs, itemIDs, "task_5_test.csv", "task_5_predictions.csv")

Guessed randomly 3441 times


In [41]:
# Task 6: compute median delta

In [42]:
a = open("task_6_train.csv")
a.readline()
userDeltas = defaultdict(list)
uLast = {} # Time of user's most recent interaction
for l in a.readlines():
    u,i,t,r = l.strip().split(',')
    t = float(t)
    if u in uLast:
        userDeltas[u].append(t - uLast[u])
    uLast[u] = t
a.close()

medians = {}
for u in userDeltas:
    medians[u] = numpy.median(userDeltas[u])

a = open("task_6_test.csv", 'r')
b = open("task_6_predictions.csv", 'w')
b.write(a.readline()) # header
for l in a.readlines():
    u,i = l.strip().split(',')
    b.write(u + ',' + i + ',' + str(medians[u]) + '\n')

b.close()

In [43]:
# Baseline performance:

# public =  {"task1": 0.7429, "task2":0.8716, "task3":0.9820, "task4":0.8036, "task5":0.8189, "task6":13563799}
# private = {"task1": 0.7738, "task2":0.8736, "task3":0.9582, "task4":0.8007, "task5":0.8195, "task6":13481806}